# EDA of the ME4OH dataset filtered to the North Atlantic Basin


In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

from plotting_functions import plot_ohc_map

## Load data

In [ ]:
df = pd.read_csv("Data/ME4OH_EN411_OFAM3/NA_1993_2014.csv")

# Convert dates to datetime, save year month and year indication columns (aids plotting)
df['Date'] = pd.to_datetime(df['Date'])
df['year_month'] = df['Date'].dt.to_period('M')
df['year'] = df['Date'].dt.to_period('Y')
df

## EDA

### Plot OHC in latitude and longitude

In [ ]:
month_list = pd.date_range('1993-1-1','2014-12-31', freq='MS').strftime('%Y-%m').tolist()
num_plots = 10
plot_months = [month_list[i] for i in range(0, len(month_list), len(month_list)//num_plots)]

In [ ]:
for n,g in df.groupby('year_month'):
    if str(n) in plot_months:
        plot_ohc_map(g, n, min(df.OHC), max(df.OHC), basin="NA")

In [ ]:
for n,g in df.groupby('year'):
    plot_ohc_map(g, n, min(df.OHC), max(df.OHC), basin="NA")

### Plot observations by platform

In [ ]:
# Filter by platforms
platforms = {
    "Argo": df.loc[df['Project name'].str.contains('ARGO')].sort_values('Date').reset_index(drop=True),
    "WOD": df.loc[df['Project name'].str.contains('WOD')].sort_values('Date').reset_index(drop=True),
    "GTSPP": df.loc[df['Project name'].str.contains('GTSPP')].sort_values('Date').reset_index(drop=True),
    "ABSO":  df.loc[df['Project name'].str.contains('ASBO')].sort_values('Date').reset_index(drop=True),
}

In [ ]:
colours = [ "#631ff3", "#ffb000", "#fe5100", "#1589e8", "#dc267f"]  # IBM colourblind palette
year_list = pd.date_range('1993-1-1','2014-12-31', freq='YS').strftime('%Y').tolist()

COUNTRIES_URL = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
WORLD = gpd.read_file(COUNTRIES_URL).rename({"ADMIN": "name"}, axis="columns")
regions = ['Southern Europe', 'Northern Africa', 'Western Africa', 'Western Europe', 'Northern Europe', 'Central America', 'South America', 'Northern America', 'Caribbean']
NA = WORLD[WORLD['SUBREGION'].isin(regions)]

In [ ]:
for year in year_list:
    _, ax = plt.subplots(figsize=(12,6))
    NA.plot(ax=ax, color='lightgray', edgecolor='gray', linewidth=0.75)  # Add countries
    for i, (name, data) in enumerate(platforms.items()):
        year_data = data.loc[data["year"] == year]
        ax.scatter(year_data.Longitude, year_data.Latitude, s=1, label=name, c=colours[i])
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_facecolor("lightblue")
    ax.set_xlim([-80, 0])
    ax.set_ylim([0, 60])
    plt.legend(bbox_to_anchor=(1,1))
    plt.title(year)
    plt.savefig(f"images/platforms_NA_by_year/platforms_NA_{year}.png")
    plt.show()
    plt.close()